# Day 1: Data Ingestion
**Bluestock Fintech — Mutual Fund Analytics Capstone**

Load all 10 raw source CSVs, inspect their structure, and validate
referential integrity (every `amfi_code` in `fund_master` must have a
matching NAV history) before any cleaning happens downstream.


In [1]:
import pandas as pd
from pathlib import Path

RAW_DIR = Path("../data/raw")

RAW_FILES = [
    "01_fund_master.csv", "02_nav_history.csv", "03_aum_by_fund_house.csv",
    "04_monthly_sip_inflows.csv", "05_category_inflows.csv", "06_industry_folio_count.csv",
    "07_scheme_performance.csv", "08_investor_transactions.csv", "09_portfolio_holdings.csv",
    "10_benchmark_indices.csv",
]

datasets = {}
for filename in RAW_FILES:
    filepath = RAW_DIR / filename
    df = pd.read_csv(filepath)
    datasets[filename] = df
    print(f"{filename:32s} shape={df.shape}")


01_fund_master.csv               shape=(40, 15)


02_nav_history.csv               shape=(46000, 3)
03_aum_by_fund_house.csv         shape=(90, 5)
04_monthly_sip_inflows.csv       shape=(48, 6)
05_category_inflows.csv          shape=(144, 3)
06_industry_folio_count.csv      shape=(21, 6)
07_scheme_performance.csv        shape=(40, 19)
08_investor_transactions.csv     shape=(32778, 13)
09_portfolio_holdings.csv        shape=(322, 8)


10_benchmark_indices.csv         shape=(8050, 3)


## Inspect fund_master

In [2]:
fund_master = datasets["01_fund_master.csv"]
print("Unique fund houses:", fund_master['fund_house'].nunique())
print("Categories:", fund_master['category'].unique())
print("Risk grades:", fund_master['risk_category'].unique())
fund_master.head()


Unique fund houses: 10
Categories: <StringArray>
['Equity', 'Debt']
Length: 2, dtype: str
Risk grades: <StringArray>
['Moderate', 'Very High', 'Low', 'High', 'Moderately High']
Length: 5, dtype: str


,amfi_code,fund_house,scheme_name,category,sub_category,plan,launch_date,benchmark,expense_ratio_pct,exit_load_pct,min_sip_amount,min_lumpsum_amount,fund_manager,risk_category,sebi_category_code
0,119551,SBI Mutual Fund,SBI Bluechip Fund - Regular Plan - Growth,Equity,Large Cap,Regular,2006-02-14,NIFTY 100 TRI,1.54,1.0,500,1000,Sohini Andani,Moderate,EC01
1,119552,SBI Mutual Fund,SBI Bluechip Fund - Direct Plan - Growth,Equity,Large Cap,Direct,2013-01-01,NIFTY 100 TRI,0.66,1.0,500,1000,Sohini Andani,Moderate,EC01
2,119598,SBI Mutual Fund,SBI Small Cap Fund - Regular Plan - Growth,Equity,Small Cap,Regular,2009-09-09,BSE 250 SmallCap TRI,1.43,1.0,500,1000,R. Srinivasan,Very High,EC03
3,119599,SBI Mutual Fund,SBI Small Cap Fund - Direct Plan - Growth,Equity,Small Cap,Direct,2013-01-01,BSE 250 SmallCap TRI,0.72,1.0,500,1000,R. Srinivasan,Very High,EC03
4,119120,SBI Mutual Fund,SBI Magnum Gilt Fund - Regular Plan - Growth,Debt,Gilt,Regular,2000-12-30,CRISIL Dynamic Gilt Index,0.77,0.0,500,1000,Dinesh Ahuja,Low,DC02


## Validate AMFI code referential integrity

In [3]:
nav_history = datasets["02_nav_history.csv"]

master_codes = set(fund_master['amfi_code'].unique())
nav_codes = set(nav_history['amfi_code'].unique())
missing_in_nav = master_codes - nav_codes

if not missing_in_nav:
    print(f"Validation OK: all {len(master_codes)} amfi_codes in fund_master have matching NAV history.")
else:
    print(f"Validation WARNING: {len(missing_in_nav)} codes missing from NAV history: {missing_in_nav}")


Validation OK: all 40 amfi_codes in fund_master have matching NAV history.


## Quick schema/dtype check on each file

In [4]:
for filename, df in datasets.items():
    print(f"\n--- {filename} ---")
    print(df.dtypes)



--- 01_fund_master.csv ---
amfi_code               int64
fund_house                str
scheme_name               str
category                  str
sub_category              str
plan                      str
launch_date               str
benchmark                 str
expense_ratio_pct     float64
exit_load_pct         float64
min_sip_amount          int64
min_lumpsum_amount      int64
fund_manager              str
risk_category             str
sebi_category_code        str
dtype: object

--- 02_nav_history.csv ---
amfi_code      int64
date             str
nav          float64
dtype: object

--- 03_aum_by_fund_house.csv ---
date                  str
fund_house            str
aum_lakh_crore    float64
aum_crore           int64
num_schemes         int64
dtype: object

--- 04_monthly_sip_inflows.csv ---
month                            str
sip_inflow_crore               int64
active_sip_accounts_crore    float64
new_sip_accounts_lakh        float64
sip_aum_lakh_crore           float64
yoy_

## Summary

All 10 raw files load successfully and pass the AMFI code integrity check.
Next step: `02_data_cleaning.ipynb` handles NAV gap-filling, transaction
validation, and loading into SQLite.
